# PIKAN prediction for the uniform infinite-domain problem

In [30]:
from pathlib import Path
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find the repository utilities directory from either the notebook folder or workspace root.
notebook_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in [notebook_dir, *notebook_dir.parents] if (path / "utils").is_dir()),
    notebook_dir,
)
utilities_dir = repo_root / "utils"
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))

import infinite
import pinns_infinite
reload(infinite)
reload(pinns_infinite)

from infinite import analytical_solution_inf, coefficient_inf, evaluate_model_inf
from pinns_infinite import build_models_KAN, set_seed, train_dual_network

set_seed(42)
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Uniform-sampling PIKAN configuration

These values match the stored uniform-infinite KAN model: three hidden layers, 25 units per layer, grid size 5, spline order 3, and uniform sampling.

In [37]:
import pandas as pd

optimization_dir = (
    repo_root
    / "main"
    / "02_hyperparameter_tunning"
    / "results_kan_infinite_optuna_2026-09-20_01-43-35"
)
summary = pd.read_csv(optimization_dir / "summary_metrics.csv")
best_trial = summary.loc[summary["mean_global_error"].idxmin()]

# Reuse the optimal hyperparameters, but train with uniform sampling.
config = {
    "mode": "load",
    "checkpoint_name": "pikan_infinite_uniform_weights.pt",
    "hidden_layers": int(best_trial["hidden_layers"]),
    "hidden_units": int(best_trial["hidden_units"]),
    "grid_size": int(best_trial["grid_size"]),
    "spline_order": int(best_trial["spline_order"]),
    "adam_lr": float(best_trial["adam_lr"]),
    "adam_iters": int(best_trial["adam_iters"]),
    "lbfgs_iters": int(best_trial["lbfgs_iters"]),
    "sampling": "uniform",
    "sigma": float(best_trial["sigma"]),
    "exp_scale": float(best_trial["exp_scale"]),
    "n_obs_u": int(best_trial["n_obs_u"]),
    "n_obs_k": int(best_trial["n_obs_k"]),
    "n_pde": int(best_trial["n_pde"]),
    "seed": int(best_trial["seed"]),
    "pde_alpha": float(best_trial["pde_alpha"]),
    "pde_beta": float(best_trial["pde_beta"]),
    "epsilon": float(best_trial["epsilon"]),
}

print("Optimal configuration extracted from hyperparameter optimization:")
print(f"Optimization source trial: {best_trial['timestamp']}")
print(f"Optimization mean global error: {best_trial['mean_global_error']:.12e}")
print("Sampling overridden for this training: uniform")
for name, value in config.items():
    print(f"{name}: {value}")

Optimal configuration extracted from hyperparameter optimization:
Optimization source trial: 2026-09-20_06-03-01
Optimization mean global error: 1.748967305617e-03
Sampling overridden for this training: uniform
mode: load
checkpoint_name: pikan_infinite_uniform_weights.pt
hidden_layers: 3
hidden_units: 15
grid_size: 5
spline_order: 4
adam_lr: 0.0001
adam_iters: 2000
lbfgs_iters: 2000
sampling: uniform
sigma: 5.5
exp_scale: 1.0
n_obs_u: 100
n_obs_k: 100
n_pde: 1000
seed: 2
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0


## Build the KAN models

The two KANs learn the solution $u(x,y)$ and variable coefficient $k(x,y)$ jointly through the physics-informed loss.

In [38]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config["hidden_layers"],
    hidden_units=config["hidden_units"],
    grid_size=config["grid_size"],
    spline_order=config["spline_order"],
)

print(model_u)
print(model_k)

KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Load the uniform-sampling PIKAN

This cell loads the stored uniform-infinite model; no retraining is performed.

In [39]:
results_dir = repo_root / "main" / "03_individual_prediction" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
weights_path = results_dir / config["checkpoint_name"]

mode = config["mode"].lower()
if mode not in {"train", "load"}:
    raise ValueError('config["mode"] must be either "train" or "load"')

if mode == "load":
    if not weights_path.exists():
        available_checkpoints = sorted(path.name for path in results_dir.glob("*.pt"))
        raise FileNotFoundError(
            f"Checkpoint not found: {weights_path}. "
            f"Available checkpoints: {available_checkpoints}"
        )

    checkpoint = torch.load(weights_path, map_location=device)
    model_u.load_state_dict(checkpoint["model_u"])
    model_k.load_state_dict(checkpoint["model_k"])
    print(f"Loaded trained model: {weights_path}")
else:
    history = train_dual_network(
        model_u,
        model_k,
        adam_lr=config["adam_lr"],
        adam_iters=config["adam_iters"],
        lbfgs_iters=config["lbfgs_iters"],
        verbose=True,
        print_every=100,
        save_every=100,
        lambda_pde_scheduler=True,
        adaptive_weights=True,
        alpha=7,
        update_every=100,
        regularization=False,
        sampling=config["sampling"],
        sigma=config["sigma"],
        exp_scale=config["exp_scale"],
        n_obs_u=config["n_obs_u"],
        n_obs_k=config["n_obs_k"],
        n_pde=config["n_pde"],
        seed=config["seed"],
        save_results=True,
        base_dir=str(results_dir),
        run_name="pikan_infinite_uniform",
        pde_alpha=config["pde_alpha"],
        pde_beta=config["pde_beta"],
        epsilon=config["epsilon"],
        device=device,
    )
    print("Training complete. Run the next cell to evaluate and save the checkpoint.")

model_u.eval()
model_k.eval()

Loaded trained model: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_infinite_uniform_weights.pt


KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)

## Evaluate against the analytical solution

In [40]:
results_dir = repo_root / "main" / "03_individual_prediction" / "results"
results_dir.mkdir(parents=True, exist_ok=True)
weights_path = results_dir / config["checkpoint_name"]

evaluation = evaluate_model_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_inf,
    coefficient=coefficient_inf,
    sampling=config["sampling"],
    train_xmin=-5.0,
    train_xmax=5.0,
    train_ymin=-5.0,
    train_ymax=5.0,
    eval_xmin=-10.0,
    eval_xmax=10.0,
    eval_ymin=-10.0,
    eval_ymax=10.0,
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

# ------------------------------------------------------------
# Evaluation metrics
# ------------------------------------------------------------

metric_names = [
    "err_u_global",
    "err_k_global",
    "err_u_inside",
    "err_k_inside",
    "err_u_outside",
    "err_k_outside",
]

metrics = {
    name: float(evaluation[name])
    for name in metric_names
}

# Mean global error
metrics["err_mean_global"] = (
    metrics["err_u_global"] +
    metrics["err_k_global"]
) / 2.0

# ------------------------------------------------------------
# Save trained model
# ------------------------------------------------------------

if config["mode"].lower() == "train":
    model_u_path = results_dir / "pikan_infinite_uniform_model_u.pt"
    model_k_path = results_dir / "pikan_infinite_uniform_model_k.pt"

    torch.save(
        {
            "model_u": model_u.state_dict(),
            "model_k": model_k.state_dict(),
            "config": config,
            "metrics": metrics,
        },
        weights_path,
    )
    torch.save(model_u.state_dict(), model_u_path)
    torch.save(model_k.state_dict(), model_k_path)

    print(f"Saved trained model: {weights_path}")
    print(f"Saved u model weights: {model_u_path}")
    print(f"Saved k model weights: {model_k_path}")

metrics


Spatial generalization (MAE)
Training domain : [-5.0, 5.0] × [-5.0, 5.0]
Evaluation domain : [-10.0, 10.0] × [-10.0, 10.0]

Global MAE
u : 1.854e-02
k : 6.506e-02

Inside training domain
u : 2.552e-03
k : 5.767e-04

Outside training domain
u : 2.387e-02
k : 8.656e-02


{'err_u_global': 0.01853809020230278,
 'err_k_global': 0.06506309651384352,
 'err_u_inside': 0.002552086228256409,
 'err_k_inside': 0.0005766882966060283,
 'err_u_outside': 0.023866758193651572,
 'err_k_outside': 0.08655856591958937,
 'err_mean_global': 0.04180059335807315}